In [1]:
# Question 1: Image Preprocessing for Inference (PyTorch) 
# Problem: Write a function to load an image and preprocess it for inference. 

from PIL import Image
import torch
from torchvision import transforms

def load_and_preprocess_image(image_path):
    # 1. Load image
    image = Image.open(image_path).convert("RGB")

    # 2. Define preprocessing
    preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    # The neural network doesn't work directly with a JPG image. It works with numbers stored in tensors.
    transforms.ToTensor(),
    transforms.Normalize(mean = [0.485, 0.456, 0.406], std = [0.229, 0.224, 0.225])])

    # 3. Apply preprocessing
    image_tensor = preprocess(image)

    # 4. Adds a batch dimension at position 0. [batch, channels, height, width]
    image_tensor = image_tensor.unsqueeze(0)

    return image_tensor

image_path = "flower.jpg"
image_tensor = load_and_preprocess_image(image_path)
print(image_tensor.shape)

torch.Size([1, 3, 224, 224])


In [2]:
# Question 2: Predict on New Image with a Trained Model 
# Problem: Perform prediction and get the class label. 

import torch
from torchvision import models

# Load pre-trained ResNet18 model
model = models.resnet18(weights = models.ResNet18_Weights.DEFAULT)

# Set model to evaluation mode
model.eval()

# Load and preprocess image
image_path ="flower.jpg"
image_tensor = load_and_preprocess_image(image_path)

# Make prediction. (During training, PyTorch needs to calculate gradients. During prediction, we don't need gradients.)
with torch.no_grad():
    output = model(image_tensor)

# Get predicted class index
predicted_class = output.argmax(dim=1).item()

print("Predicted class index:", predicted_class)


weights = models.ResNet18_Weights.DEFAULT

categories = weights.meta["categories"]

predicted_label = categories[predicted_class]

print("Predicted class label:", predicted_label)

Predicted class index: 985
Predicted class label: daisy


In [3]:
# Question 3: Build a CNN to classify CIFAR-10 images (PyTorch) 
# Problem: Create a CNN model that classifies images from the CIFAR-10 dataset with accuracy above 60%. 

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Preprocessing (ToTensor() converts the image into a PyTorch tensor. Normalize() scales the pixel values so that training becomes easier.)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,)*3, (0.5,)*3)
])

# Download CIFAR-10
train_data = datasets.CIFAR10("./data", train=True, download=True, transform=transform)
test_data = datasets.CIFAR10("./data", train=False, download=True, transform=transform)
train_data = torch.utils.data.Subset(train_data, range(10000))

# Create DataLoaders (The dataset is divided into batches of 64 images.)
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=64)

# CNN 
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(128*4*4, 10)
        )

    def forward(self, x):
        return self.net(x)

# Create the model. Loss function and optimizer
model = CNN()
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training
for epoch in range(5):
    model.train()
    for x, y in train_loader:
        optimizer.zero_grad()
        loss = loss_fn(model(x), y)
        loss.backward()
        optimizer.step()

# Accuracy
model.eval()
correct = total = 0

with torch.no_grad():
    for x, y in test_loader:
        pred = model(x).argmax(1)
        correct += (pred == y).sum().item()
        total += y.size(0)

print("Accuracy:", 100 * correct / total, "%")

Accuracy: 60.39 %


In [4]:
# Question 4: Identify Overfitting from Training Logs and Solve It 
# Problem: You notice the training accuracy increases but validation accuracy stagnates. 
# Modify the model using dropout and early stopping. (use mnist dataset) 

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

data = datasets.MNIST("./data", train=True, download=True, transform=transforms.ToTensor())

data = torch.utils.data.Subset(data, range(10000))
# Split into training and validation data
train, val = random_split(data, [8000, 2000])
train_loader = DataLoader(train, 64, shuffle=True)
val_loader = DataLoader(val, 64)

# Create CNN model with Dropout
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Conv2d(1, 32, 3), nn.ReLU(), nn.MaxPool2d(2),
                                 nn.Flatten(),
                                 nn.Linear(32*13*13, 128), nn.ReLU(),
                                 # Dropout to reduce overfitting
                                 nn.Dropout(0.5),
                                 nn.Linear(128, 10))
    # This tells PyTorch how the input image should pass through the CNN.
    def forward(self, x):
        return self.net(x)

model = CNN()
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 'best' stores the best validation accuracy seen so far. 'wait' counts how many consecutive epochs have passed without improvement.
best = wait = 0 

# Train the model
for epoch in range(5):
    model.train()
    for x, y in train_loader:
        # Clears the gradients from the previous batch
        optimizer.zero_grad()
        loss_fn(model(x), y).backward()
        # Updates the model's weights using the calculated gradients.
        optimizer.step()
    # Validate the model
    model.eval()
    with torch.no_grad():
        for x, y in val_loader:
            correct = sum((model(x).argmax(1) == y).sum().item()
                      for x, y in val_loader)

    acc = correct / len(val) * 100
    print(f"Epoch {epoch+1}: {acc:.2f}%")

    if acc > best:
        best, wait = acc, 0
    else:
        wait += 1
        if wait == 2:
            print("Early stopping!")
            break

Epoch 1: 90.90%
Epoch 2: 93.80%
Epoch 3: 94.20%
Epoch 4: 95.50%
Epoch 5: 95.75%


In [1]:
# Question 5: Transfer Learning with Pretrained VGG16 (Cats vs Dogs) 
# Problem: Use VGG16 for binary classification with fine-tuning 

import tensorflow as tf
from tensorflow.keras.applications import VGG16
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model

train_folder = "train"
validation_folder = "validation"
test_folder = "test"

train_data = tf.keras.utils.image_dataset_from_directory(
    train_folder, image_size=(224,224), batch_size=32)

val_data = tf.keras.utils.image_dataset_from_directory(
    validation_folder, image_size=(224,224), batch_size=32)

test_data = tf.keras.utils.image_dataset_from_directory(
    test_folder, image_size=(224,224), batch_size=32, shuffle=False)

base_model = VGG16(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

base_model.trainable = False

x = GlobalAveragePooling2D()(base_model.output)
x = Dense(128, activation="relu")(x)
output = Dense(2, activation="softmax")(x)

model = Model(inputs=base_model.input, outputs=output)

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=1
)

test_loss, test_accuracy = model.evaluate(test_data)

print("Test Accuracy:", test_accuracy)

Found 19943 files belonging to 2 classes.
Found 2492 files belonging to 2 classes.
Found 2495 files belonging to 2 classes.
624/624 ━━━━━━━━━━━━━━━━━━━━ 2119s 3s/step - accuracy: 0.9597 - loss: 0.1393 - val_accuracy: 0.9671 - val_loss: 0.0893
78/78 ━━━━━━━━━━━━━━━━━━━━ 240s 3s/step - accuracy: 0.9635 - loss: 0.1111
Test Accuracy: 0.9635270833969116


In [8]:
# Question 5
# This will find images that don't have 1, 3, or 4 channels and remove them.
import os
from PIL import Image

folders = ["train", "validation", "test"]

for folder in folders:
    print("\nChecking:", folder)

    for root, dirs, files in os.walk(folder):
        for file in files:
            if file.lower().endswith((".jpg", ".jpeg", ".png")):
                path = os.path.join(root, file)

                try:
                    img = Image.open(path)

                    # Check image channels
                    if img.mode not in ["L", "RGB", "RGBA"]:
                        print("Removing:", path, "Mode:", img.mode)
                        os.remove(path)

                except Exception:
                    print("Removing corrupted image:", path)
                    os.remove(path)

print("\nDataset checking completed!")


Checking: train

Checking: validation

Checking: test

Dataset checking completed!


In [10]:
# Question 5
# convert every image to RGB and overwrite it
import os
from PIL import Image

folders = ["train", "validation", "test"]

for folder in folders:
    print("\nProcessing:", folder)

    for root, dirs, files in os.walk(folder):
        for file in files:
            if file.lower().endswith((".jpg", ".jpeg", ".png")):
                path = os.path.join(root, file)

                try:
                    img = Image.open(path)
                    img.load()

                    # Convert every image to RGB
                    img = img.convert("RGB")
                    img.save(path)

                except Exception as e:
                    print("Removing bad image:", path)
                    os.remove(path)

print("\nAll images are now RGB!")


Processing: train

Processing: validation

Processing: test

All images are now RGB!
